# 데이터 분할

In [6]:
# 네이버 영화 리뷰(NSMC) 분할 스크립트
# - 입력 : ratings_train.txt (id, document, label)
# - 출력 : nsmc_train.tsv / nsmc_valid.tsv / nsmc_test.tsv

import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

# 1) 데이터 로드
df = pd.read_csv("data/ratings_train.txt", sep="\t").dropna(subset=["document", "label"])
df["label"] = df["label"].astype(int)

# 2) (권장) 완전 중복 제거(문장 기준)
before = len(df)
df = df.drop_duplicates(subset=["document"]).reset_index(drop=True)
after = len(df)
print(f"중복 제거: {before} -> {after} (제거 {before-after})")

# 3) 라벨 분포 확인 함수
def label_ratio(s):
    return s.value_counts(normalize=True).sort_index().round(4)

print("전체 라벨 비율:", label_ratio(df["label"]).to_dict())

# 4) Stratified 분할: 8:1:1  (Train 80%, Valid 10%, Test 10%)
X = df["document"]
y = df["label"]

# 먼저 test 10% 홀드아웃
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=42
)

# 남은 90%에서 valid 10/90 = 전체의 10%가 되도록 분리
X_train, X_valid, y_train, y_valid = train_test_split(
    X_tmp, y_tmp, test_size=0.1111, stratify=y_tmp, random_state=42
)  # 0.1111 ≈ 1/9

print("Train 라벨 비율:", label_ratio(y_train).to_dict())
print("Valid 라벨 비율:", label_ratio(y_valid).to_dict())
print("Test  라벨 비율:", label_ratio(y_test).to_dict())

# 5) 저장(추후 재현/배포를 위해 TSV 권장)
pd.DataFrame({"document": X_train, "label": y_train}).to_csv("nsmc_train.tsv", sep="\t", index=False)
pd.DataFrame({"document": X_valid, "label": y_valid}).to_csv("nsmc_valid.tsv", sep="\t", index=False)
pd.DataFrame({"document": X_test,  "label": y_test }).to_csv("nsmc_test.tsv",  sep="\t", index=False)

print("Saved: nsmc_train.tsv / nsmc_valid.tsv / nsmc_test.tsv")

# ------------------------------------------------------------
# (옵션) 교차검증용 StratifiedKFold 폴드 인덱스 만들기 (train만 대상으로)
# - 모델/하이퍼 파라미터 탐색 시 사용
# StratifiedKFold는 각 Fold가 전체 데이터의 라벨 분포를 동일하게 유지하도록 분할
# ------------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_tr = X_train.reset_index(drop=True)
y_tr = y_train.reset_index(drop=True)

folds = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr, y_tr)):
    folds.append({"fold": fold,
                  "train_idx": tr_idx.tolist(),
                  "valid_idx": va_idx.tolist()})
print(f"StratifiedKFold(5-fold) 생성 완료. (예: folds[0]['train_idx'] …)")

# 참고:
# - 벡터라이저/스케일러는 반드시 "train으로 fit → valid/test에는 transform만" 적용하세요.
# - NSMC에는 user_id/movie_id가 없으므로 Group Split은 적용 불가합니다.


중복 제거: 149995 -> 146182 (제거 3813)
전체 라벨 비율: {0: 0.5017, 1: 0.4983}
Train 라벨 비율: {0: 0.5017, 1: 0.4983}
Valid 라벨 비율: {0: 0.5017, 1: 0.4983}
Test  라벨 비율: {0: 0.5017, 1: 0.4983}
Saved: nsmc_train.tsv / nsmc_valid.tsv / nsmc_test.tsv
StratifiedKFold(5-fold) 생성 완료. (예: folds[0]['train_idx'] …)


In [16]:
label_ratio(y_tr[folds[0]['train_idx']]).to_dict()

{0: 0.5017, 1: 0.4983}

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# 예제 데이터프레임 생성
data = {
    'document': [
        "이 영화 정말 재미있다", "최고였어요", "정말 별로였다", "지루하고 재미없음",
        "감동적이다", "다시 보고 싶다", "시간 낭비", "최악의 경험"
    ],
    'label': [1, 1, 0, 0, 1, 1, 0, 0],       # 긍정(1), 부정(0)
    'user_id': ['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D']  # 같은 사용자가 여러 리뷰 작성
}

df = pd.DataFrame(data)

X = df['document']
y = df['label']
groups = df['user_id']

# StratifiedGroupKFold 설정
sgkf = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=42)

for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    print(f"\n===== Fold {fold_idx} =====")
    print("Train Index:", train_idx)
    print("Test Index :", test_idx)
    
    print("\nTrain 데이터:")
    print(df.iloc[train_idx])
    
    print("\nTest 데이터:")
    print(df.iloc[test_idx])


# 데이터 토큰화

In [ ]:
# 공백 기반 토큰화
text = "나는 학교에 간다"
tokens = text.split()
print(tokens)  # ['나는', '학교에', '간다']


['나는', '학교에', '간다']


In [19]:
from konlpy.tag import Okt

okt = Okt()
text = "나는 학교에 간다"
print(okt.morphs(text))
# ['나', '는', '학교', '에', '간다']

print(okt.pos(text))
# [('나', 'Noun'), ('는', 'Josa'), ('학교', 'Noun'), ('에', 'Josa'), ('간다', 'Verb')]


['나', '는', '학교', '에', '간다']
[('나', 'Noun'), ('는', 'Josa'), ('학교', 'Noun'), ('에', 'Josa'), ('간다', 'Noun')]


In [36]:
import pandas as pd

# NSMC 학습 데이터 로드
df = pd.read_csv("data/ratings_train.txt", sep="\t").dropna(subset=["document"])
texts = df["document"].astype(str).head(3).tolist()  # 앞 3개만 샘플
for i, t in enumerate(texts, 1):
    print(f"[샘플 {i}] {t}")


[샘플 1] 아 더빙.. 진짜 짜증나네요 목소리
[샘플 2] 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
[샘플 3] 너무재밓었다그래서보는것을추천한다


In [37]:
from konlpy.tag import Okt

okt = Okt()

def okt_tokenize(text: str):
    return okt.morphs(text if isinstance(text, str) else "")

print("\n=== OKT 토큰화 ===")
for i, t in enumerate(texts, 1):
    print(f"[샘플 {i}]", okt_tokenize(t))



=== OKT 토큰화 ===
[샘플 1] ['아', '더빙', '..', '진짜', '짜증나네요', '목소리']
[샘플 2] ['흠', '...', '포스터', '보고', '초딩', '영화', '줄', '....', '오버', '연기', '조차', '가볍지', '않구나']
[샘플 3] ['너', '무재', '밓었', '다그', '래서', '보는것을', '추천', '한', '다']


In [ ]:
# !pip install sentencepiece

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 16.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import pandas as pd

df_tr = pd.read_csv("data/ratings_train.txt", sep="\t").dropna(subset=["document"])
df_te = pd.read_csv("data/ratings_test.txt",  sep="\t").dropna(subset=["document"])
corpus = pd.concat([df_tr["document"], df_te["document"]], ignore_index=True)

# 아주 가벼운 정리 (필수 아님)
corpus = corpus.str.replace(r"\s+", " ", regex=True).str.strip()

corpus.to_csv("corpus.txt", index=False, header=False)
print("saved corpus.txt with", len(corpus), "lines")


saved corpus.txt with 199992 lines


In [29]:
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
    input="corpus.txt",         # 학습에 사용할 텍스트 파일 (한 줄에 하나의 문장)
    model_prefix="ko_unigram",  # 출력 파일명 prefix → ko_unigram.model, ko_unigram.vocab 생성
    vocab_size=16000,           # 단어사전 크기 (자주 8k, 16k, 32k 등 사용)  
                                # 작으면 일반화는 좋지만 세밀한 표현이 부족할 수 있음
                                # 크면 성능은 좋지만 모델 크기가 커지고 학습 속도 느려짐
    model_type="unigram",       # 토큰 생성 방식 (unigram, bpe, word, char)
                                # - 'unigram': BERT, KoGPT 등에서 사용하는 언어모델 기반 방식 (한국어에 매우 적합)
                                # - 'bpe': Byte Pair Encoding, GPT-2 등에서 사용
                                # - 'word': 단어 단위 (한국어에는 적합하지 않음)
                                # - 'char': 문자 단위 (정보 너무 잘게 쪼개짐)
    character_coverage=0.9995,  # 학습에 포함할 문자 종류의 비율
                                # - 1.0: 모든 문자 포함 (일본어, 중국어, 한글 포함)
                                # - 한국어는 0.9995 권장 → 한글, 영어, 숫자 대부분 포함
    input_sentence_size=1000000,# 학습 문장을 샘플링할 양 (random sampling)
                                # corpus.txt 전체를 학습하면 시간이 오래 걸리므로
                                # 일부만 샘플링하여 빠르게 학습하는 것이 일반적
    shuffle_input_sentence=True,# 샘플링 시 문장의 순서를 섞어서 학습
                                # 모델이 특정 순서에 편향되지 않도록 랜덤화
    # pad_id=0,                   # <pad>의 ID (Padding token, 기본값 0)
    # unk_id=1,                   # <unk>의 ID (Unknown token = 사전에 없는 문자)
    # bos_id=2,                   # <s> (문장의 시작 토큰)
    # eos_id=3                    # </s> (문장의 끝 토큰)
                                # ※ bos_id, eos_id는 옵션이며, sequence-to-sequence 모델에서 주로 사용
)
# 생성: ko_unigram.model / ko_unigram.vocab


| 토큰      | 목적       | NLP 모델에서의 핵심 역할            |
| ------- | -------- | -------------------------- |
| `<pad>` | 길이 맞추기   | 배치 단위 연산 가능하게 함            |
| `<unk>` | 미지 단어 처리 | 어떤 단어라도 입력을 막지 않음 (모델 안정성) |
| `<s>`   | 문장 시작 명시 | Seq2Seq, GPT 모델 입력의 시작 신호  |
| `</s>`  | 문장 끝 명시  | 모델이 생성 종료를 이해하도록 도움        |


In [ ]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load("ko_unigram.model")  # 사전 학습 완료된 모델 로드
text = "나는 학교에 간다"
print(sp.encode(text, out_type=str))
# ['▁나는', '▁학교에', '▁간', '다']


['▁나는', '▁학교', '에', '▁간다']


In [31]:
char = "\u2581"
print(char)  # 출력: ▁

▁


In [38]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor(model_file="ko_unigram.model")

def sp_tokenize(text: str):
    return sp.encode(text if isinstance(text, str) else "", out_type=str)

print("\n=== SentencePiece 토큰화 ===")
for i, t in enumerate(texts, 1):
    print(f"[샘플 {i}]", sp_tokenize(t))
# 참고: '▁'는 단어 시작(공백)을 의미하는 특수표시



=== SentencePiece 토큰화 ===
[샘플 1] ['▁아', '▁더빙', '..', '▁진짜', '▁짜증나', '네요', '▁목소리']
[샘플 2] ['▁흠', '...', '포스터', '보고', '▁초딩', '영화', '줄', '....', '오', '버', '연기', '조차', '▁가볍지', '▁않', '구나']
[샘플 3] ['▁너무', '재', '밓', '었다', '그래서', '보는것', '을', '추천', '한다']


In [ ]:
# !pip install JPype1 konlpy


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


| 태그  | 설명    | 예시      |
| --- | ----- | ------- |
| NNG | 일반 명사 | 영화, 학교  |
| NNP | 고유 명사 | 서울, 아이유 |
| NNB | 의존 명사 | 것, 수, 데 |
| NP  | 대명사   | 나, 너, 저 |
| NR  | 수사    | 하나, 둘   |

----

| 태그  | 설명     | 예시      |
| --- | ------ | ------- |
| VV  | 동사     | 가다, 먹다  |
| VA  | 형용사    | 크다, 예쁘다 |
| VX  | 보조 용언  | 않다, 못하다 |
| VCP | 긍정 지정사 | 이다      |
| VCN | 부정 지정사 | 아니다     |
---
| 태그  | 설명    | 예시       | 포함 여부         |
| --- | ----- | -------- | ------------- |
| MAG | 일반 부사 | 매우, 정말   | 상황에 따라 포함     |
| MAJ | 접속 부사 | 그리고, 그러나 | 맥락 분석 시 포함    |
| MM  | 관형사   | 이, 그, 저  | 일반적으로 제외      |
| IC  | 감탄사   | 아!, 어머!  | 감정 분석 시 포함 가능 |
---
| 태그  | 설명     | 예시   | 포함 여부 |
| --- | ------ | ---- | ----- |
| JKS | 주격 조사  | 이/가  | ❌ 제외  |
| JKC | 보격 조사  | 이/가  | ❌ 제외  |
| JKG | 관형격 조사 | 의    | ❌ 제외  |
| JKO | 목적격 조사 | 을/를  | ❌ 제외  |
| JX  | 보조사    | 도, 만 | ❌ 제외  |
| JC  | 접속 조사  | 와/과  | ❌ 제외  |
| EP  | 선어말 어미 | 었, 겠 | ❌ 제외  |
| EF  | 어말 어미  | 다, 요 | ❌ 제외  |
| EC  | 연결 어미  | 고, 면 | ❌ 제외  |
---
| 태그 | 설명   | 예시      | 포함 여부      |
| -- | ---- | ------- | ---------- |
| SN | 숫자   | 123     | 업무에 따라 선택  |
| SL | 외국어  | AI, ML  | 포함 추천      |
| SH | 한자   | 學, 愛    | 데이터에 따라 다름 |
| SF | 마침표  | . , ! ? | 제외         |
| SW | 기타기호 | ~, @    | 제외         |


### 감성/의도 분석/리뷰 분석용 (가장 일반적)
- "NNG", "NNP", "VV", "VA", "MAG", "SL"
### 명사 기반 분류 (문서 주제 분류 전용)
- "NNG", "NNP", "NR", "NP"
### 의미 있는 단어만 최대한 포함하고 싶을 때
- "NNG", "NNP", "VV", "VA", "MAG", "MAJ", "IC", "SL"

In [34]:
from konlpy.tag import Komoran

komoran = Komoran()  # 기본 사전

text = "나는 학교에 간다"

print("morphs :", komoran.morphs(text))   # 형태소 나열
print("pos    :", komoran.pos(text))      # (형태소, 품사) 튜플
print("nouns  :", komoran.nouns(text))    # 명사만 추출


morphs : ['나', '는', '학교', '에', '간다']
pos    : [('나', 'NP'), ('는', 'JX'), ('학교', 'NNG'), ('에', 'JKB'), ('간다', 'NNP')]
nouns  : ['학교', '간다']


In [39]:
from konlpy.tag import Komoran

komoran = Komoran()

def komoran_tokenize(text: str):
    return [m for m in komoran.morphs(text if isinstance(text, str) else "")]

print("\n=== Komoran 토큰화 ===")
for i, t in enumerate(texts, 1):
    print(f"[샘플 {i}]", komoran_tokenize(t))



=== Komoran 토큰화 ===
[샘플 1] ['아', '더빙', '.', '.', '진짜', '짜증', '나', '네요', '목소리']
[샘플 2] ['흠', '...', '포스터', '보고', '초딩', '영화', '줄', '....', '오버', '연기', '조차', '가볍', '지', '않', '구나']
[샘플 3] ['너무재밓었다그래서보는것을추천한다']


In [40]:
# Komoran 예: 명사/동사/형용사만 남기기
ALLOW_POS = {"NNG","NNP","VV","VA"}  # 일반명사/고유명사/동사/형용사
from konlpy.tag import Komoran
komoran = Komoran()

def komoran_tokenize_filtered(text: str):
    toks = []
    for morph, pos in komoran.pos(text if isinstance(text, str) else ""):
        if pos in ALLOW_POS:
            toks.append(morph)
    return toks

print("\n=== Komoran(품사 필터) ===")
for i, t in enumerate(texts, 1):
    print(f"[샘플 {i}]", komoran_tokenize_filtered(t))



=== Komoran(품사 필터) ===
[샘플 1] ['더빙', '짜증', '나', '목소리']
[샘플 2] ['포스터', '초딩', '영화', '오버', '연기', '가볍']
[샘플 3] []


# 백터화

In [60]:
import pandas as pd

# NSMC 데이터 로드
df = pd.read_csv("data/ratings_train.txt", sep="\t").dropna(subset=["document"])

# 상위 5개의 문장만 샘플로 사용
texts = df["document"].astype(str).head(10).tolist()

print("📌 샘플 문장:")
for i, text in enumerate(texts, 1):
    print(f"{i}. {text}")


📌 샘플 문장:
1. 아 더빙.. 진짜 짜증나네요 목소리
2. 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
3. 너무재밓었다그래서보는것을추천한다
4. 교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
5. 사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다
6. 막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.
7. 원작의 긴장감을 제대로 살려내지못했다.
8. 별 반개도 아깝다 욕나온다 이응경 길용우 연기생활이몇년인지..정말 발로해도 그것보단 낫겟다 납치.감금만반복반복..이드라마는 가족도없다 연기못하는사람만모엿네
9. 액션이 없는데도 재미 있는 몇안되는 영화
10. 왜케 평점이 낮은건데? 꽤 볼만한데.. 헐리우드식 화려함에만 너무 길들여져 있나?


In [43]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer를 이용한 One-Hot Encoding (binary=True)
vectorizer = CountVectorizer(binary=True)  # 단어 존재 여부만 표시

X = vectorizer.fit_transform(texts)  # 학습 및 변환

# 단어 사전 출력
vocab = vectorizer.get_feature_names_out()
print("\n📚 단어 사전 (Vocabulary):")
print(vocab)

# One-Hot 벡터 출력
print("\n🔥 One-Hot 인코딩 결과 (문장 → 벡터):")
print(X.toarray())



📚 단어 사전 (Vocabulary):
['가볍지' '교도소' '너무나도' '너무재밓었다그래서보는것을추천한다' '늙어보이기만' '더빙' '던스트가' '돋보였던' '목소리'
 '사이몬페그의' '솔직히' '스파이더맨에서' '않구나' '없다' '연기가' '영화' '오버연기조차' '이뻐보였다' '이야기구먼'
 '익살스런' '재미는' '조정' '진짜' '짜증나네요' '초딩영화줄' '커스틴' '평점' '포스터보고' '했던']

🔥 One-Hot 인코딩 결과 (문장 → 벡터):
[[0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0 1 1 0 0 0 0 1 0 0]
 [0 0 1 0 1 0 1 1 0 1 0 1 0 0 1 1 0 1 0 1 0 0 0 0 0 1 0 0 1]]


In [44]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import CountVectorizer

okt = Okt()

def okt_tokenize(text: str):
    # 기본 형태소 토큰화 (필요하면 stopwords 제거/품사 필터 추가 가능)
    return okt.morphs(text if isinstance(text, str) else "")

vectorizer_okt = CountVectorizer(
    tokenizer=okt_tokenize,
    lowercase=False,   # 한국어 소문자화 의미 적음
    binary=True        # ← One-Hot (존재 여부만 1/0)
)

X_okt = vectorizer_okt.fit_transform(texts)
print("\n[OKT] 단어 사전:", vectorizer_okt.get_feature_names_out())
print("[OKT] One-Hot 행렬 shape:", X_okt.shape)
print(X_okt.toarray())



[OKT] 단어 사전: ['!' '..' '...' '....' '가' '가볍지' '교도소' '구먼' '그' '너' '너무나도' '는' '늙어' '다'
 '다그' '더빙' '던스트' '돋보였던' '래서' '목소리' '몬페' '무재' '밓었' '보고' '보는것을' '보였다' '보이기만'
 '사이' '솔직히' '스파이더맨' '아' '않구나' '없다' '에서' '연기' '영화' '오버' '의' '이뻐' '이야기'
 '익살스런' '재미' '조정' '조차' '줄' '진짜' '짜증나네요' '초딩' '추천' '커스틴' '평점' '포스터' '한'
 '했던' '흠']
[OKT] One-Hot 행렬 shape: (5, 55)
[[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0]
 [0 0 1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 1
  1 0 0 0 0 0 0 1 1 0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 1 1 0 0 0 1 0 0 1 1 0 1 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0]
 [0 1 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0
  0 0 0 1 0 1 1 0 0 0 0 0 0 0 1 0 0 0 0]
 [1 0 0 0 1 0 0 0 1 0 1 0 1 0 0 0 1 1 0 0 1 0 0 0 0 1 1 1 0 1 0 0 0 1 1 1
  0 1 1 0 1 0 0 0 0 0 0 0 0 1 0 0 0 1 0]]


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [45]:
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import CountVectorizer

komoran = Komoran()

# 의미 위주 품사만 남기는 간단 필터 (명사/동사/형용사/부사/외국어)
ALLOW_POS = {"NNG","NNP","VV","VA","MAG","SL"}
STOPWORDS = {"하다","되다"}  # 필요시 확장

def komoran_tokenize(text: str):
    if not isinstance(text, str): 
        return []
    toks = []
    for morph, pos in komoran.pos(text):
        if pos in ALLOW_POS and morph not in STOPWORDS:
            toks.append(morph)
    return toks

vectorizer_komo = CountVectorizer(
    tokenizer=komoran_tokenize,
    lowercase=False,
    binary=True        # ← One-Hot
)

X_komo = vectorizer_komo.fit_transform(texts)
print("\n[Komoran] 단어 사전:", vectorizer_komo.get_feature_names_out())
print("[Komoran] One-Hot 행렬 shape:", X_komo.shape)
print(X_komo.toarray())



[Komoran] 단어 사전: ['가볍' '교도소' '나' '너무나' '늙' '더빙' '돋보이' '목소리' '보이' '솔직히' '스파이더맨' '없' '연기'
 '영화' '오버' '이야기' '익살' '재미' '조정' '진짜' '짜증' '초딩' '커스틴 던스트' '평점' '포스터' '하']
[Komoran] One-Hot 행렬 shape: (5, 26)
[[0 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 1 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 1 1 0 0 0 0 1 0 0]
 [0 0 0 1 1 0 1 0 1 0 1 0 1 1 0 0 1 0 0 0 0 0 1 0 0 1]]


In [ ]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor(model_file="ko_unigram.model")

def sp_tokenize(text: str):
    # SentencePiece는 공백을 '▁'(U+2581)로 표시 → 그대로 둘지 제거할지 선택 가능
    # 그대로 쓰면 토큰이 '▁자연어' 형태, 제거하려면 아래 1줄 주석 해제
    toks = sp.encode(text if isinstance(text, str) else "", out_type=str)
    # toks = [t.replace("▁", "") for t in toks]  # 단어 시작표시 제거하려면 사용
    return toks


=== SentencePiece 토큰 예시 ===
[샘플 1] ['▁아', '▁더빙', '..', '▁진짜', '▁짜증나', '네요', '▁목소리']
[샘플 2] ['▁흠', '...', '포스터', '보고', '▁초딩', '영화', '줄', '....', '오', '버', '연기', '조차', '▁가볍지', '▁않', '구나']
[샘플 3] ['▁너무', '재', '밓', '었다', '그래서', '보는것', '을', '추천', '한다']
[샘플 4] ['▁교도소', '▁이야기', '구먼', '▁', '..', '솔직히', '▁재미는', '▁없다', '..', '평점', '▁조정']
[샘플 5] ['▁사이', '몬', '페', '그', '의', '▁', '익', '살', '스런', '▁연기가', '▁돋보였던', '▁영화', '!', '스파이더맨', '에서', '▁늙어', '보이', '기만', '▁했던', '▁커', '스틴', '▁', '던', '스트', '가', '▁너무나도', '▁이뻐', '보', '였다']


# 📘 CountVectorizer 매개변수 정리 (scikit-learn)

`CountVectorizer`는 문서를 단어 단위로 토큰화한 후, 각 단어의 등장 횟수를 기반으로 벡터화합니다.

---

## 🔧 1. 기본 파라미터 (입력/전처리 관련)

| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| **input** | 'content' | 입력 형식 ('filename', 'file', 'content') |
| **encoding** | 'utf-8' | 텍스트 인코딩 방식 |
| **decode_error** | 'strict' | 디코딩 오류 처리 ('strict', 'ignore', 'replace') |
| **strip_accents** | None | 악센트 제거 옵션 ('ascii', 'unicode') |
| **lowercase** | True | 모든 텍스트를 소문자로 변환 |
| **preprocessor** | None | 사용자 정의 전처리 함수 적용 |
| **tokenizer** | None | 사용자 정의 토큰화 함수 (형태소 분석기 연결 시 사용) |
| **stop_words** | None | 불용어 제거 ('english' 또는 리스트) |

---

## 🔤 2. 단어 추출(Tokens) 제어 관련

| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| **token_pattern** | r'(?u)\\b\\w\\w+\\b' | 2글자 이상 단어만 추출 |
| **ngram_range** | (1,1) | n-gram 범위 설정 (예: (1,2) → unigram + bigram) |
| **analyzer** | 'word' | 'word', 'char', 'char_wb' 선택 가능 |
| **max_df** | 1.0 | 너무 자주 등장하는 단어 제거 (비율 또는 정수) |
| **min_df** | 1 | 너무 적게 등장하는 단어 제거 (비율 또는 정수) |
| **max_features** | None | 상위 n개의 단어만 사용 (차원 제한) |
| **vocabulary** | None | 단어 사전을 고정하거나 직접 지정 |

---

## 📊 3. 출력 벡터 관련 설정

| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| **binary** | False | `True`시 단어 등장 여부(1/0)만 기록 (빈도 무시) |
| **dtype** | `np.int64` | 출력 데이터 타입 |

---


## 🧠 주요 속성 (Attributes)

| 속성 | 설명 |
|------|------|
| **vocabulary_** | 단어:인덱스 매핑 딕셔너리 |
| **stop_words_** | 제거된 불용어 리스트 |
| **ngram_range** | 적용된 n-gram 범위 |

---

In [47]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_sp = CountVectorizer(
    tokenizer=sp_tokenize,
    lowercase=False,
    binary=True,          # ← One-Hot처럼 '존재 여부'만 1/0
    min_df=1             # 데모용(실전은 2~5로 노이즈 감소)
)

X_sp = vectorizer_sp.fit_transform(texts)

print("\n📚 Vocabulary 크기:", len(vectorizer_sp.get_feature_names_out()))
print("📚 일부 Vocabulary:", vectorizer_sp.get_feature_names_out()[:30])
print("\n🔥 One-Hot 행렬 shape:", X_sp.shape)
print(X_sp.toarray())



📚 Vocabulary 크기: 67
📚 일부 Vocabulary: ['!' '..' '...' '....' '▁' '▁가볍지' '▁교도소' '▁너무' '▁너무나도' '▁늙어' '▁더빙' '▁돋보였던'
 '▁목소리' '▁사이' '▁아' '▁않' '▁없다' '▁연기가' '▁영화' '▁이뻐' '▁이야기' '▁재미는' '▁조정' '▁진짜'
 '▁짜증나' '▁초딩' '▁커' '▁했던' '▁흠' '가']

🔥 One-Hot 행렬 shape: (5, 67)
[[0 1 0 0 0 0 0 0 0 0 1 0 1 0 1 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 1
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 0 0 0
  0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 0 0 0 1 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0
  0 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 1]
 [0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 0 0 0 0
  0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [1 0 0 0 1 0 0 0 1 1 0 1 0 1 0 0 0 1 1 1 0 0 0 0 0 0 1 1 0 1 0 0 1 0 1 0
  1 1 0 0 1 0 0 1 1 0 1 1 1 1 0 1 0 1 0 0 0 1 1 0 0 0 0 1 0 0 0]]


# 📘 TfidfVectorizer 매개변수 정리 (scikit-learn)

`TfidfVectorizer`는 **단어의 빈도(TF)**와 **희소성(IDF)**을 결합해 문서에서 단어의 중요도를 수치화하는 벡터화 도구입니다.

---

## 🔧 1. 입력 및 전처리 관련 파라미터

| 매개변수 | 기본값 | 설명 |
|----------|--------|------|
| **input** | 'content' | 입력 타입 ('content', 'filename', 'file') |
| **encoding** | 'utf-8' | 파일/문자열 인코딩 방식 |
| **decode_error** | 'strict' | 디코딩 오류 처리 방식 ('strict', 'ignore', 'replace') |
| **strip_accents** | None | 악센트 제거 ('ascii', 'unicode' 가능) |
| **lowercase** | True | 모든 텍스트를 소문자로 변환 |
| **preprocessor** | None | 사용자 정의 전처리 함수 지정 |
| **tokenizer** | None | 사용자 정의 토큰화 함수 (형태소 분석기 연결 시 사용) |
| **analyzer** | 'word' | 분석 단위 ('word', 'char', 'char_wb') |
| **stop_words** | None | 불용어 제거 ('english' 또는 사용자 정의 리스트) |

---

## 🧮 2. 토큰화 및 단어 선택 관련 파라미터

| 매개변수 | 기본값 | 설명 |
|----------|--------|------|
| **token_pattern** | r'(?u)\\b\\w\\w+\\b' | 2글자 이상인 단어만 추출 |
| **ngram_range** | (1,1) | n-gram 범위 설정 (예: (1,2)는 unigram + bigram) |
| **max_df** | 1.0 | 상위 문서 비율로 제거 (예: 0.8 → 80% 이상 문서에 등장 시 제거) |
| **min_df** | 1 | 최소 등장 문서 수 또는 비율 (예: 2 → 최소 2개 문서에 등장해야 포함) |
| **max_features** | None | 상위 N개의 단어만 사용 (차원 제한) |
| **vocabulary** | None | 단어 사전을 직접 지정 가능 (dict 또는 list) |

---

## 📊 3. TF-IDF 계산 관련 파라미터

| 매개변수 | 기본값 | 설명 |
|----------|--------|------|
| **norm** | 'l2' | 벡터 정규화 방식 ('l1', 'l2', None) |
| **use_idf** | True | IDF 사용 여부 (False 시 단순 TF만 사용) |
| **smooth_idf** | True | IDF 계산 시 분모가 0되지 않도록 +1 적용 |
| **sublinear_tf** | False | TF 값에 로그 스케일 적용 (`1 + log(tf)`) |

---

## ⚙ 출력 및 데이터 타입 관련

| 매개변수 | 기본값 | 설명 |
|----------|--------|------|
| **dtype** | float64 | 벡터 값의 데이터 타입 (float32 가능) |

---

## 🧠 주요 속성 (Attributes)

| 속성 | 설명 |
|------|------|
| **vocabulary_** | 단어-인덱스 매핑 |
| **idf_** | 각 단어의 IDF 값 배열 |
| **stop_words_** | 제거된 불용어 목록 |
| **fixed_vocabulary_** | vocabulary가 고정되었는지 여부 |

---

## 📌 주요 메서드 (Methods)

| 메서드 | 설명 |
|--------|------|
| **fit(raw_documents)** | 단어 사전 + IDF 학습 |
| **transform(raw_documents)** | TF-IDF 행렬 변환 |
| **fit_transform(raw_documents)** | fit + transform 동시에 수행 |
| **get_feature_names_out()** | TF-IDF 기준으로 추출된 단어 리스트 |
| **inverse_transform(X)** | 벡터를 다시 단어 목록으로 변환 |

---

In [52]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer

okt = Okt()

def okt_tokenize(text):
    return okt.morphs(text)

vectorizer_okt = TfidfVectorizer(
    tokenizer=okt_tokenize,
    ngram_range=(1,2),
    min_df=1,
    lowercase=False
)

X_okt = vectorizer_okt.fit_transform(texts)
vocab_okt = vectorizer_okt.get_feature_names_out()

print("\n=== ✅ OKT 기반 TF-IDF Vocabulary ===")
print(vocab_okt)

print("\n=== ✅ OKT 기반 TF-IDF 행렬 ===")
print(X_okt.toarray())



=== ✅ OKT 기반 TF-IDF Vocabulary ===
['!' '! 스파이더맨' '..' '.. 솔직히' '.. 진짜' '.. 평점' '...' '... 포스터' '....'
 '.... 오버' '가' '가 너무나도' '가 돋보였던' '가볍지' '가볍지 않구나' '교도소' '교도소 이야기' '구먼'
 '구먼 ..' '그' '그 의' '너' '너 무재' '너무나도' '너무나도 이뻐' '는' '는 없다' '늙어' '늙어 보이기만'
 '다' '다그' '다그 래서' '더빙' '더빙 ..' '던스트' '던스트 가' '돋보였던' '돋보였던 영화' '래서'
 '래서 보는것을' '목소리' '몬페' '몬페 그' '무재' '무재 밓었' '밓었' '밓었 다그' '보고' '보고 초딩' '보는것을'
 '보는것을 추천' '보였다' '보이기만' '보이기만 했던' '사이' '사이 몬페' '솔직히' '솔직히 재미' '스파이더맨'
 '스파이더맨 에서' '아' '아 더빙' '않구나' '없다' '없다 ..' '에서' '에서 늙어' '연기' '연기 가' '연기 조차'
 '영화' '영화 !' '영화 줄' '오버' '오버 연기' '의' '의 익살스런' '이뻐' '이뻐 보였다' '이야기' '이야기 구먼'
 '익살스런' '익살스런 연기' '재미' '재미 는' '조정' '조차' '조차 가볍지' '줄' '줄 ....' '진짜'
 '진짜 짜증나네요' '짜증나네요' '짜증나네요 목소리' '초딩' '초딩 영화' '추천' '추천 한' '커스틴' '커스틴 던스트'
 '평점' '평점 조정' '포스터' '포스터 보고' '한' '한 다' '했던' '했던 커스틴' '흠' '흠 ...']

=== ✅ OKT 기반 TF-IDF 행렬 ===
[[0.         0.         0.2472117  0.         0.30641253 0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.

In [53]:
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer

komoran = Komoran()

# 의미 있는 품사만 선택 (명사/동사/형용사/부사)
ALLOW_POS = {"NNG","NNP","VV","VA","MAG","SL"}
STOPWORDS = {"하다", "되다"}

def komoran_tokenize(text):
    tokens = []
    for morph, pos in komoran.pos(text):
        if pos in ALLOW_POS and morph not in STOPWORDS:
            tokens.append(morph)
    return tokens

vectorizer_komoran = TfidfVectorizer(
    tokenizer=komoran_tokenize,
    ngram_range=(1,2),
    min_df=1,
    lowercase=False
)

X_komoran = vectorizer_komoran.fit_transform(texts)
vocab_komoran = vectorizer_komoran.get_feature_names_out()

print("\n=== ✅ Komoran 기반 TF-IDF Vocabulary ===")
print(vocab_komoran)

print("\n=== ✅ Komoran 기반 TF-IDF 행렬 ===")
print(X_komoran.toarray())



=== ✅ Komoran 기반 TF-IDF Vocabulary ===
['가볍' '교도소' '교도소 이야기' '나' '나 목소리' '너무나' '늙' '늙 보이' '더빙' '더빙 진짜' '돋보이'
 '돋보이 영화' '목소리' '보이' '보이 하' '솔직히' '솔직히 재미' '스파이더맨' '스파이더맨 늙' '없' '없 평점'
 '연기' '연기 가볍' '연기 돋보이' '영화' '영화 스파이더맨' '영화 오버' '오버' '오버 연기' '이야기'
 '이야기 솔직히' '익살' '익살 연기' '재미' '재미 없' '조정' '진짜' '진짜 짜증' '짜증' '짜증 나' '초딩'
 '초딩 영화' '커스틴 던스트' '커스틴 던스트 너무나' '평점' '평점 조정' '포스터' '포스터 초딩' '하'
 '하 커스틴 던스트']

=== ✅ Komoran 기반 TF-IDF 행렬 ===
[[0.         0.         0.         0.33333333 0.33333333 0.
  0.         0.         0.33333333 0.33333333 0.         0.
  0.33333333 0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.33333333 0.33333333 0.33333333 0.33333333 0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.31156077 0.         0.         0.         0.         0.
  0. 

In [54]:
import sentencepiece as spm
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer

# (1) SentencePiece 모델 준비 (없으면 자동 학습)
MODEL = Path("ko_unigram.model")
if not MODEL.exists():
    with open("corpus.txt", "w", encoding="utf-8") as f:
        for t in texts:
            f.write(t + "\n")
    spm.SentencePieceTrainer.Train(
        input="corpus.txt",
        model_prefix="ko_unigram",
        vocab_size=2000,
        character_coverage=0.9995,
        model_type="unigram"
    )

# (2) 모델 로드
sp = spm.SentencePieceProcessor(model_file="ko_unigram.model")

def sp_tokenize(text):
    return sp.encode(text if isinstance(text, str) else "", out_type=str)

vectorizer_sp = TfidfVectorizer(
    tokenizer=sp_tokenize,
    ngram_range=(1,2),
    min_df=1,
    lowercase=False
)

X_sp = vectorizer_sp.fit_transform(texts)
vocab_sp = vectorizer_sp.get_feature_names_out()

print("\n=== ✅ SentencePiece 기반 TF-IDF Vocabulary ===")
print(vocab_sp)

print("\n=== ✅ SentencePiece 기반 TF-IDF 행렬 ===")
print(X_sp.toarray())



=== ✅ SentencePiece 기반 TF-IDF Vocabulary ===
['!' '! 스파이더맨' '..' '.. ▁진짜' '.. 솔직히' '.. 평점' '...' '... 포스터' '....'
 '.... 오' '▁' '▁ ..' '▁ 던' '▁ 익' '▁가볍지' '▁가볍지 ▁않' '▁교도소' '▁교도소 ▁이야기' '▁너무'
 '▁너무 재' '▁너무나도' '▁너무나도 ▁이뻐' '▁늙어' '▁늙어 보이' '▁더빙' '▁더빙 ..' '▁돋보였던'
 '▁돋보였던 ▁영화' '▁목소리' '▁사이' '▁사이 몬' '▁아' '▁아 ▁더빙' '▁않' '▁않 구나' '▁없다'
 '▁없다 ..' '▁연기가' '▁연기가 ▁돋보였던' '▁영화' '▁영화 !' '▁이뻐' '▁이뻐 보' '▁이야기' '▁이야기 구먼'
 '▁재미는' '▁재미는 ▁없다' '▁조정' '▁진짜' '▁진짜 ▁짜증나' '▁짜증나' '▁짜증나 네요' '▁초딩' '▁초딩 영화'
 '▁커' '▁커 스틴' '▁했던' '▁했던 ▁커' '▁흠' '▁흠 ...' '가' '가 ▁너무나도' '구나' '구먼' '구먼 ▁'
 '그' '그 의' '그래서' '그래서 보는것' '기만' '기만 ▁했던' '네요' '네요 ▁목소리' '던' '던 스트' '몬'
 '몬 페' '밓' '밓 었다' '버' '버 연기' '보' '보 였다' '보고' '보고 ▁초딩' '보는것' '보는것 을' '보이'
 '보이 기만' '살' '살 스런' '솔직히' '솔직히 ▁재미는' '스런' '스런 ▁연기가' '스트' '스트 가' '스틴'
 '스틴 ▁' '스파이더맨' '스파이더맨 에서' '었다' '었다 그래서' '에서' '에서 ▁늙어' '연기' '연기 조차' '였다'
 '영화' '영화 줄' '오' '오 버' '을' '을 추천' '의' '의 ▁' '익' '익 살' '재' '재 밓' '조차'
 '조차 ▁가볍지' '줄' '줄 ....' '추천' '추천 한다' '페' '페 그' '평점' '평점 ▁조정' '포스터'
 '포스터 보고' '한다']

=== ✅ Sentence

| 방식                | 특징               | 장점         | 단점        |
| ----------------- | ---------------- | ---------- | --------- |
| **Okt**           | 형태소 분석 + 띄어쓰기 기반 | 간단, 직관적    | 속도 느림     |
| **Komoran**       | 품사 기반 정교한 분리     | 정확도, 속도 우수 | Java 필요   |
| **SentencePiece** | subword 단위 분리    | OOV 강함     | 형태소 정보 없음 |


In [ ]:
!pip install gensim

# 📘 Word2Vec 매개변수 정리 (gensim.models.Word2Vec)

Word2Vec은 **CBOW**와 **Skip-Gram** 두 가지 방식으로 단어의 의미를 벡터 공간에 학습시키는 모델입니다.

---

## 🔧 주요 매개변수(Parameter)

| 매개변수 | 기본값 | 설명 |
|----------|--------|------|
| **sentences** | None | 토큰화된 문장 리스트 (필수 입력) |
| **vector_size** | 100 | 임베딩 차원 수. 높을수록 표현력 ↑, 계산량 ↑ |
| **window** | 5 | 학습 시 주변 단어를 몇 개까지 고려할지 (문맥 크기) |
| **min_count** | 5 | 등장 빈도가 min_count 미만인 단어는 무시 |
| **sg** | 0 | 학습 방식 선택: 0 = CBOW, 1 = Skip-Gram |
| **hs** | 0 | 히에라키컬 소프트맥스(1) 사용 여부 (기본은 Negative Sampling) |
| **negative** | 5 | 네거티브 샘플링 개수 (0이면 사용 안 함) |
| **epochs** | 5 | 전체 학습 반복 횟수 |
| **alpha** | 0.025 | 초기 학습률 |
| **min_alpha** | 0.0001 | 최소 학습률 (linear decay) |
| **seed** | 1 | 랜덤 시드 (재현성 확보용) |
| **workers** | 3 | 병렬 처리에 사용할 CPU 코어 수 |
| **compute_loss** | False | 학습 중 손실(loss) 기록 여부 |
| **max_final_vocab** | None | 최종 단어 사전의 최대 크기 제한 |
| **sample** | 0.001 | 빈도 높은 단어를 다운샘플링 (과대표현 방지) |
| **sorted_vocab** | 1 | 단어 사전 정렬 여부 (빈도순) |
| **trim_rule** | None | 사용자 정의 단어 필터링 규칙 |

---

## 🧠 Word2Vec의 핵심 파라미터 요약

| 파라미터 | 역할 | 영향 |
|----------|------|------|
| `sg` | 0=CBOW(빠름), 1=Skip-Gram(의미 정확도 ↑) | 의미 학습의 방향 결정 |
| `vector_size` | 임베딩 크기 | 크면 성능↑, 너무 크면 과적합/메모리↑ |
| `window` | 문맥 범위 | 작으면 국소적 의미, 크면 전체 주제 기반 의미 |
| `min_count` | 학습 단어 제한 | 노이즈 필터링 |
| `negative` | negative sampling 개수 | 학습 속도와 성능의 균형 |
| `hs` | Hierarchical Softmax 사용 여부 | 희귀 단어 학습에 유리 |
| `epochs` | 학습 반복 수 | 언더/오버피팅 방지 |

---

In [61]:
from konlpy.tag import Okt
from gensim.models import Word2Vec

okt = Okt()

# OKT 토큰화 함수
def okt_tokenizer(text):
    return okt.morphs(text)

# 전체 문장을 토큰화
tokenized_okt = [okt_tokenizer(sentence) for sentence in texts]
print("📌 OKT 토큰화 예:", tokenized_okt[0])

# Word2Vec 학습
model_okt = Word2Vec(
    sentences=tokenized_okt,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1  # Skip-gram 방식
)

# 단어 벡터와 유사도 확인
print("\n🎯 OKT Word2Vec 유사 단어:", model_okt.wv.most_similar("영화", topn=5))


📌 OKT 토큰화 예: ['아', '더빙', '..', '진짜', '짜증나네요', '목소리']

🎯 OKT Word2Vec 유사 단어: [('평점', 0.17826755344867706), ('도', 0.1312638223171234), ('반복', 0.07491309195756912), ('이', 0.06765002757310867), ('연기', 0.041578300297260284)]


In [62]:
from konlpy.tag import Komoran
from gensim.models import Word2Vec

komoran = Komoran()
ALLOW_POS = {"NNG", "NNP", "VV", "VA", "MAG", "SL"}

def komoran_tokenizer(text):
    tokens = []
    for morph, pos in komoran.pos(text):
        if pos in ALLOW_POS:
            tokens.append(morph)
    return tokens

tokenized_komoran = [komoran_tokenizer(sentence) for sentence in texts]
print("\n📌 Komoran 토큰화 예:", tokenized_komoran[0])

model_komoran = Word2Vec(
    sentences=tokenized_komoran,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1
)

print("\n🎯 Komoran Word2Vec 유사 단어:", model_komoran.wv.most_similar("영화", topn=5))



📌 Komoran 토큰화 예: ['더빙', '진짜', '짜증', '나', '목소리']

🎯 Komoran Word2Vec 유사 단어: [('이', 0.1701882779598236), ('평점', 0.145950585603714), ('아깝', 0.06408977508544922), ('재미', -0.002754019573330879), ('반복', -0.01351492665708065)]


In [67]:
import sentencepiece as spm
from gensim.models import Word2Vec

sp = spm.SentencePieceProcessor(model_file="ko_unigram.model")

def sp_tokenizer(text):
    return sp.encode(text, out_type=str)  # '▁단어' 형식 유지 (단어 시작 표시 포함)

tokenized_sp = [sp_tokenizer(sentence) for sentence in texts]
print("\n📌 SentencePiece 토큰화 예:", tokenized_sp[0])

model_sp = Word2Vec(
    sentences=tokenized_sp,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1
)

print("\n🎯 SentencePiece Word2Vec 유사 단어:", model_sp.wv.most_similar('▁영화', topn=5))

word_vector = model_sp.wv["영화"]
print(word_vector.mean())  # "영화"라는 단어의 100차원 벡터



📌 SentencePiece 토큰화 예: ['▁아', '▁더빙', '..', '▁진짜', '▁짜증나', '네요', '▁목소리']

🎯 SentencePiece Word2Vec 유사 단어: [('▁너무', 0.12819039821624756), ('살', 0.10943625867366791), ('이', 0.10876553505659103), ('..', 0.06277808547019958), ('반복', 0.05049639940261841)]
0.0011008705


#### 유사도
의미적으로 가까운 단어들끼리 유사한 방향(또는 가까운 위치)

## 모델 학습

In [70]:
# ===========================
# 0. 필수 패키지 설치 (없으면 설치)
# ===========================
# pip install konlpy JPype1 pandas scikit-learn gensim xgboost sentencepiece transformers

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
try:
    from xgboost import XGBClassifier
    has_xgb = True
except:
    print("⚠️ xgboost 설치 필요: pip install xgboost")
    has_xgb = False

# ===========================
# 1. NSMC 데이터 로드
# ===========================
df = pd.read_csv("data/ratings_train.txt", sep="\t").dropna(subset=["document", "label"])
df = df.drop_duplicates(subset=["document"])  # 중복 제거
X = df["document"].astype(str).values
y = df["label"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("✅ 데이터 준비 완료:", X_train.shape, X_test.shape)

# ===========================
# 2. 토크나이저 정의
# ===========================

# 🔹 2-1) OKT
from konlpy.tag import Okt
okt = Okt()
def okt_tokenize(text):
    return okt.morphs(text)

# 🔹 2-2) Komoran
from konlpy.tag import Komoran
komo = Komoran()
ALLOW_POS = {"NNG","NNP","VV","VA","MAG","SL"}
STOPWORDS = {"하다","되다"}
def komoran_tokenize(text):
    tokens = []
    for morph, pos in komo.pos(text):
        if pos in ALLOW_POS and morph not in STOPWORDS:
            tokens.append(morph)
    return tokens

# 🔹 2-3) SentencePiece
import os, sentencepiece as spm
from pathlib import Path

MODEL = Path("ko_sp.model")
if not MODEL.exists():
    print("🚀 SentencePiece 모델 학습 시작 (최초 1회)")
    with open("sp_corpus.txt", "w", encoding="utf-8") as f:
        for t in X_train:
            f.write(t + "\n")
    spm.SentencePieceTrainer.Train(
        input="sp_corpus.txt",
        model_prefix="ko_sp",
        vocab_size=8000,
        character_coverage=0.9995,
        model_type="unigram"
    )
    print("✅ SentencePiece 모델 학습 완료")

sp = spm.SentencePieceProcessor(model_file="ko_sp.model")
def sp_tokenize(text):
    return sp.encode(text, out_type=str)

# ===========================
# 3. TF-IDF + 분류 모델 구성 함수
# ===========================
def run_models(name, tokenizer):

    vectorizer = TfidfVectorizer(
        tokenizer=tokenizer,
        ngram_range=(1,2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
        lowercase=False
    )
    
    models = {
        "LogReg": LogisticRegression(max_iter=1000, C=2.0, class_weight="balanced", n_jobs=-1),
        "SVM": LinearSVC(C=1.0),
        "RandomForest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    }
    if has_xgb:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            n_jobs=-1,
            random_state=42,
            tree_method="hist"
        )
    
    print(f"\n\n==============================")
    print(f"🔷 토큰화 방식: {name}")
    print(f"==============================")

    for model_name, model in models.items():
        pipe = Pipeline([
            ("tfidf", vectorizer),
            ("clf", model)
        ])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1m = f1_score(y_test, y_pred, average="macro")
        print(f"\n▶ 모델: {model_name}")
        print(f"   - Accuracy: {acc:.4f}")
        print(f"   - F1-macro: {f1m:.4f}")

# ===========================
# 4. 실행 (3개 토크나이저 비교)
# ===========================
run_models("OKT", okt_tokenize)
run_models("Komoran", komoran_tokenize)
run_models("SentencePiece", sp_tokenize)


✅ 데이터 준비 완료: (116945,) (29237,)
🚀 SentencePiece 모델 학습 시작 (최초 1회)
✅ SentencePiece 모델 학습 완료


🔷 토큰화 방식: OKT


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(



▶ 모델: LogReg
   - Accuracy: 0.8643
   - F1-macro: 0.8643


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(



▶ 모델: SVM
   - Accuracy: 0.8586
   - F1-macro: 0.8586


...


▶ 모델: SVM
   - Accuracy: 0.8147
   - F1-macro: 0.8147


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(



▶ 모델: RandomForest
   - Accuracy: 0.8073
   - F1-macro: 0.8073


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


KeyboardInterrupt: 

In [72]:
# TF-IDF + Linear SVM (Pipeline + GridSearchCV)
# 파일: ratings_train.txt (컬럼: id, document, label)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

RANDOM_STATE = 42

# 1) 데이터 로드 & 기본 정리
df = pd.read_csv("data/ratings_train.txt", sep="\t").dropna(subset=["document","label"])
df = df.drop_duplicates(subset=["document"]).reset_index(drop=True)  # 완전 중복 문장 제거
X = df["document"].astype(str).values
y = df["label"].astype(int).values

# 2) Stratified 분할(라벨 비율 유지)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# 3) 파이프라인 정의: TF-IDF → Linear SVM
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),     # uni+bi 기본
        min_df=3,              # 너무 희귀한 토큰 제거
        max_df=0.95,           # 너무 흔한 토큰 제거
        sublinear_tf=True,     # tf를 log(1+tf)로 스케일
        lowercase=False        # 한국어는 소문자화 무의미
    )),
    ("clf", LinearSVC())
])

# 4) 그리드 탐색 공간
param_grid = {
    "tfidf__ngram_range": [(1,1), (1,2)],
    "tfidf__min_df": [2, 3, 5],
    "tfidf__max_df": [0.9, 0.95, 1.0],
    "clf__C": [0.5, 1.0, 2.0, 4.0]
}

# 5) 교차검증 설정(계층화)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# 6) GridSearchCV
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1_macro",    # 불균형에 조금 더 견고
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# 7) 학습(훈련셋으로만)
grid.fit(X_tr, y_tr)

print("\n===== Grid Search 결과 =====")
print("Best params :", grid.best_params_)
print("Best CV F1  :", f"{grid.best_score_:.4f}")

# 8) 테스트 평가(훈련 외 검증셋)
best_model = grid.best_estimator_
y_pred = best_model.predict(X_te)

print("\n===== Test 성능 =====")
print("Accuracy :", f"{accuracy_score(y_te, y_pred):.4f}")
print("F1-macro :", f"{f1_score(y_te, y_pred, average='macro'):.4f}")
print("\nClassification Report\n", classification_report(y_te, y_pred, digits=3))

cm = confusion_matrix(y_te, y_pred)
print("Confusion Matrix\n", cm)

# (옵션) 피처 수 확인
tfidf = best_model.named_steps["tfidf"]
print("\nVocabulary size:", len(tfidf.vocabulary_))


Fitting 5 folds for each of 72 candidates, totalling 360 fits

===== Grid Search 결과 =====
Best params : {'clf__C': 0.5, 'tfidf__max_df': 0.9, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}
Best CV F1  : 0.8063

===== Test 성능 =====
Accuracy : 0.8139
F1-macro : 0.8137

Classification Report
               precision    recall  f1-score   support

           0      0.798     0.843     0.820     14669
           1      0.832     0.784     0.808     14568

    accuracy                          0.814     29237
   macro avg      0.815     0.814     0.814     29237
weighted avg      0.815     0.814     0.814     29237

Confusion Matrix
 [[12367  2302]
 [ 3140 11428]]

Vocabulary size: 96610


In [74]:
!pip install datasets

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ---------------------------------------- 0.0/26.2 MB ? eta -:--:--
   ----- ---------------------------------- 3.7/26.2 MB 18.1 MB/s eta 0:00:02
   ----------------- ---------------------- 11.8/26.2 MB 29.5 MB/s eta 0:00:01
   ---------------------------------------  26.0/26.2 MB 42.2 MB/s eta 0:00:01
   ---------------------------------------- 26.2/26.2 MB 40.5 MB/s eta 0:00:00

  Attempting uninstall: pyarrow

    Found existing installation: pyarrow 20.0.0

    Uninstalling pyarrow-20.0.0:

      Successfully uninstalled pyarrow-20.0.0

   -- -------------------------------------  1/15 [pyarrow]
   -- -------------------------------------  1/15 [pyarrow]
   -- -------------------------------------  1/15 [pyarrow]
   -- ----------------------------------

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install Korpora


   -------------------- ------------------- 1/2 [Korpora]
   -------------------- ------------------- 1/2 [Korpora]
   ---------------------------------------- 2/2 [Korpora]




[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
